<a href="https://colab.research.google.com/github/sancode24/BNS-RAG-legal-assistant/blob/main/nhanes_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NHANES diabetes project — data pipeline

Run top to bottom. Output: `nhanes_out/nhanes_all.parquet` (SEQN, cycle, period, w, stratum, psu, labels, subgroups).

**Do not use the data downstream until cell 4b says VALIDATION PASSED.**

## 0. Setup

In [1]:
# NHANES diabetes project — data pipeline (Sanjana)
# Output: nhanes_out/nhanes_all.parquet with SEQN, cycle, period, w, stratum, psu,
#         labels (y_diag, y_bio, y_bio_fpg, screened) and subgroup columns.
# Raw NHANES column names are kept until Rushmitha's crosswalk is ready.
import os

import numpy as np
import pandas as pd
import requests
from scipy import stats

# Optional (recommended): save everything to Google Drive so it survives resets
# from google.colab import drive; drive.mount('/content/drive')
# os.makedirs('/content/drive/MyDrive/nhanes_project', exist_ok=True)
# os.chdir('/content/drive/MyDrive/nhanes_project')

RAW_DIR, OUT_DIR = "nhanes_raw", "nhanes_out"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# cycle -> (URL year, file prefix, file suffix, legacy folder, period)
CYCLES = {
    "2011-2012": (2011, "",   "_G", "2011-2012", "train"),
    "2013-2014": (2013, "",   "_H", "2013-2014", "train"),
    "2015-2016": (2015, "",   "_I", "2015-2016", "train"),
    "2017-2020": (2017, "P_", "",   "2017-2018", "test_P"),  # pre-pandemic file; never split it
    "2021-2023": (2021, "",   "_L", "2021-2022", "test_L"),
}
TRAIN_CYCLES = [c for c, v in CYCLES.items() if v[4] == "train"]

# file -> columns to keep (None = keep every column). Any column starting with "WT" is always kept.
FILES = {
    "DEMO":   ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "DMDEDUC2", "INDFMPIR",
               "RIDEXPRG", "SDMVSTRA", "SDMVPSU"],
    "DIQ":    ["SEQN", "DIQ010", "DIQ160", "DIQ180", "DIQ050", "DIQ070"],
    "HIQ":    ["SEQN", "HIQ011"],
    "HUQ":    ["SEQN", "HUQ030"],
    "INQ":    None,
    "BMX":    ["SEQN", "BMXBMI", "BMXWAIST"],
    "GHB":    ["SEQN", "LBXGH"],
    "GLU":    ["SEQN", "LBXGLU"],
    "FASTQX": ["SEQN", "PHAFSTHR"],
    "BP":     None,   # BPX (2011-2016, auscultatory) or BPXO (2017+, oscillometric)
}
BP_FILE = {"2011-2012": "BPX", "2013-2014": "BPX", "2015-2016": "BPX",
           "2017-2020": "BPXO", "2021-2023": "BPXO"}

HEADERS = {"User-Agent": "Mozilla/5.0 (research download script)"}

## 1. Download

In [2]:
def file_name(base, cycle):
    _, prefix, suffix, _, _ = CYCLES[cycle]
    if base == "BP":
        base = BP_FILE[cycle]
    return f"{prefix}{base}{suffix}"


def candidate_urls(name, cycle):
    year, _, _, legacy, _ = CYCLES[cycle]
    return [
        f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{name}.xpt",
        f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{name}.XPT",
        f"https://wwwn.cdc.gov/Nchs/Nhanes/{legacy}/{name}.XPT",
    ]


def download(base, cycle):
    """Download one NHANES .xpt file (cached). Returns local path or None."""
    name = file_name(base, cycle)
    path = os.path.join(RAW_DIR, f"{name}.xpt")
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    for url in candidate_urls(name, cycle):
        try:
            r = requests.get(url, headers=HEADERS, timeout=60)
        except requests.RequestException:
            continue
        if r.status_code == 200 and r.content[:13] == b"HEADER RECORD":   # real SAS file, not an error page
            with open(path, "wb") as f:
                f.write(r.content)
            return path
    print(f"  !! could not download {name} ({cycle}) — copy the link from the NHANES data page")
    return None


def read_xpt(path, wanted):
    df = pd.read_sas(path, format="xport", encoding="latin-1")
    df.columns = [c.upper() for c in df.columns]
    if wanted is None:
        return df
    keep = [c for c in wanted if c in df.columns] + [c for c in df.columns if c.startswith("WT")]
    return df[list(dict.fromkeys(keep))]


raw = {}
for cycle in CYCLES:
    print(f"Downloading {cycle} ...")
    for base, cols in FILES.items():
        p = download(base, cycle)
        if p:
            raw[(base, cycle)] = read_xpt(p, cols)
print("Done.")

Done.


## 2. Merge and clean special codes

In [3]:
def merge_cycle(cycle):
    if ("DEMO", cycle) not in raw:
        raise RuntimeError(f"DEMO missing for {cycle}")
    df = raw[("DEMO", cycle)].copy()
    for base in FILES:
        if base == "DEMO" or (base, cycle) not in raw:
            continue
        other = raw[(base, cycle)]
        other = other[[c for c in other.columns if c == "SEQN" or c not in df.columns]]
        df = df.merge(other, on="SEQN", how="left")
    df["cycle"] = cycle
    df["period"] = CYCLES[cycle][4]
    df["bp_method"] = "oscillometric" if BP_FILE[cycle] == "BPXO" else "auscultatory"
    return df


df = pd.concat([merge_cycle(c) for c in CYCLES if ("DEMO", c) in raw], ignore_index=True)
df["SEQN"] = df["SEQN"].astype("int64")

check_vars = ["DIQ010", "DIQ180", "HIQ011", "HUQ030", "LBXGH", "LBXGLU", "BMXBMI", "INDFMPIR"]
avail = df.groupby("cycle")[check_vars].apply(lambda g: g.notna().mean().round(3))
print("Share of non-missing values per cycle (0 = variable absent in that cycle):")
print(avail.to_string())

SPECIAL = [7, 9, 77, 99, 777, 999, 7777, 9999]          # Refused / Don't know -> missing
for col in ["DIQ010", "DIQ160", "DIQ180", "DIQ050", "DIQ070", "HIQ011", "HUQ030", "DMDEDUC2"]:
    if col in df:
        df[col] = df[col].where(~df[col].isin(SPECIAL))

Share of non-missing values per cycle (0 = variable absent in that cycle):
           DIQ010  DIQ180  HIQ011  HUQ030  LBXGH  LBXGLU  BMXBMI  INDFMPIR
cycle                                                                     
2011-2012   0.960   0.628   1.000     1.0  0.630   0.311   0.882     0.914
2013-2014   0.960   0.636   1.000     1.0  0.653   0.312   0.890     0.923
2015-2016   0.960   0.620   1.000     1.0  0.634   0.298   0.878     0.894
2017-2020   0.963   0.630   1.000     1.0  0.626   0.305   0.844     0.859
2021-2023   0.984   0.696   0.998     1.0  0.563   0.308   0.710     0.829


## 3. Weights, design, labels, subgroups, period

In [4]:
def first_present(frame, names):
    for n in names:
        if n in frame.columns:
            return frame[n]
    return pd.Series(np.nan, index=frame.index)


# --- weights (use the weight of the smallest subsample the variables come from)
df["w_int"] = first_present(df, ["WTINT2YR"]).fillna(first_present(df, ["WTINTPRP"]))
df["w_mec"] = first_present(df, ["WTMEC2YR"]).fillna(first_present(df, ["WTMECPRP"]))
# FIX 1: HbA1c in 2021-23 uses the phlebotomy weight WTPH2YR (0 = no blood sample)
if "WTPH2YR" not in df.columns and "2021-2023" in df["cycle"].values:
    raise KeyError("WTPH2YR not found for 2021-2023 — check DEMO_L / GHB_L codebooks")
df["w_lab"] = np.where(df["cycle"].eq("2021-2023"), first_present(df, ["WTPH2YR"]), df["w_mec"])
fast_cols = [c for c in df.columns if c.startswith("WTSAF")]      # WTSAF2YR / WTSAFPRP
df["w_fast"] = df[fast_cols].bfill(axis=1).iloc[:, 0] if fast_cols else np.nan
for wcol in ["w_mec", "w_lab"]:
    if df.groupby("cycle")[wcol].apply(lambda s: s.isna().all()).any():
        raise KeyError(f"{wcol} is entirely missing for at least one cycle — check weight names")

# --- pooled weights for analyses that combine cycles (CDC rule: 2-yr cycles each get 2/total years)
POOL_FACTOR = {c: (2 / 6 if c in TRAIN_CYCLES else 1.0) for c in CYCLES}
df["pool_factor"] = df["cycle"].map(POOL_FACTOR)
df["w"] = df["w_lab"] * df["pool_factor"]            # main analysis weight (HbA1c-based labels)
df["w_fast_pooled"] = df["w_fast"] * df["pool_factor"]

# --- design: strata/PSU numbers repeat across cycles -> make them unique
df["stratum"] = df["cycle"] + "_" + df["SDMVSTRA"].astype("Int64").astype(str)
df["psu"] = df["stratum"] + "_" + df["SDMVPSU"].astype("Int64").astype(str)

# --- analysis population: adults 20+, not pregnant
pregnant = df["RIDEXPRG"].eq(1) if "RIDEXPRG" in df else False
df["adult"] = (df["RIDAGEYR"] >= 20) & ~pregnant

# --- labels (names agreed with Rushmitha's labels.py)
df["y_diag"] = df["DIQ010"].map({1: 1.0, 2: 0.0, 3: 0.0})       # borderline (3) is not a diagnosis
a1c_pos = df["LBXGH"] >= 6.5
ok = df["y_diag"].notna() & df["LBXGH"].notna()
df["y_bio"] = np.where(ok, ((df["y_diag"] == 1) | a1c_pos).astype(float), np.nan)        # PRIMARY, weight w
df["undx"] = np.where(ok, ((df["y_diag"] == 0) & a1c_pos).astype(float), np.nan)

# y_bio_fpg: sensitivity label, ONLY on the fasting subsample (8-24 h fast), weight w_fast
fasted = df["PHAFSTHR"].between(8, 24) if "PHAFSTHR" in df else True
ok_f = ok & df["LBXGLU"].notna() & (df["w_fast"].fillna(0) > 0) & fasted
bio_f = a1c_pos | (df["LBXGLU"] >= 126)
df["y_bio_fpg"] = np.where(ok_f, ((df["y_diag"] == 1) | bio_f).astype(float), np.nan)
df["undx_fpg"] = np.where(ok_f, ((df["y_diag"] == 0) & bio_f).astype(float), np.nan)
df["y_diag_fpg"] = np.where(ok_f, df["y_diag"], np.nan)          # diagnosed, same fasting population

# screened: DIQ180, asked only of people WITHOUT a diagnosis (DIQ159 is the routing check item)
df["screened"] = np.where(df["y_diag"].eq(0), df["DIQ180"].map({1: 1.0, 2: 0.0}), np.nan)

# --- subgroups (cut points: confirm on day 1)
df["income_band"] = pd.cut(df["INDFMPIR"], [-np.inf, 1.3, 3.5, np.inf],
                           labels=["<1.3", "1.3-3.5", ">3.5"], right=False)
df["insured"] = df["HIQ011"].map({1: 1.0, 2: 0.0})
df["routine_care"] = df["HUQ030"].map({1: 1.0, 3: 1.0, 2: 0.0})
df["age_band"] = pd.cut(df["RIDAGEYR"], [20, 40, 60, np.inf], labels=["20-39", "40-59", "60+"], right=False)
df["sex"] = df["RIAGENDR"].map({1: "male", 2: "female"})
df["race_eth"] = df["RIDRETH3"].map({1: "Mexican American", 2: "Other Hispanic", 3: "NH White",
                                     4: "NH Black", 6: "NH Asian", 7: "Other/Multi"})

CORE = ["SEQN", "cycle", "period", "w", "stratum", "psu", "w_int", "w_mec", "w_lab", "w_fast",
        "w_fast_pooled", "pool_factor", "adult", "bp_method",
        "y_diag", "y_bio", "undx", "y_bio_fpg", "undx_fpg", "y_diag_fpg", "screened",
        "income_band", "insured", "routine_care", "age_band", "sex", "race_eth"]
raw_cols = [c for c in df.columns if c not in CORE and c not in ("SDMVSTRA", "SDMVPSU")]
nhanes_all = df[CORE + raw_cols]

try:
    nhanes_all.to_parquet(os.path.join(OUT_DIR, "nhanes_all.parquet"), index=False)
    print(f"Saved {len(nhanes_all):,} rows -> {OUT_DIR}/nhanes_all.parquet")
except Exception as e:   # pyarrow missing
    nhanes_all.to_csv(os.path.join(OUT_DIR, "nhanes_all.csv"), index=False)
    print(f"Parquet failed ({e}); saved CSV instead")

Saved 57,395 rows -> nhanes_out/nhanes_all.parquet


## 4. Survey-weighted prevalence: Taylor SE + t-based CI (design df)

In [5]:
def svy_prop(d, y, w, domain=None):
    """Weighted proportion of binary y for ONE cycle; domain rows are zeroed, not dropped."""
    in_dom = (d[w].fillna(0) > 0) & d[y].notna()
    if domain is not None:
        in_dom &= domain
    wt = np.where(in_dom, d[w].fillna(0), 0.0)
    yv = np.where(in_dom, d[y].fillna(0), 0.0)
    W = wt.sum()
    if W == 0:
        return np.nan, np.nan, 0
    p = (wt * yv).sum() / W
    z = wt * (yv - p) / W
    tot = pd.DataFrame({"h": d["stratum"].values, "psu": d["psu"].values, "z": z}) \
            .groupby(["h", "psu"])["z"].sum().reset_index()
    var = 0.0
    for _, g in tot.groupby("h"):
        n = len(g)
        if n > 1:
            var += n / (n - 1) * ((g["z"] - g["z"].mean()) ** 2).sum()
    return p, np.sqrt(var), int(in_dom.sum())


MEASURES = [  # (label, variable, weight, extra domain)
    ("Diagnosed (y_diag)",               "y_diag",     "w_lab",  None),
    ("Total, HbA1c (y_bio)",             "y_bio",      "w_lab",  None),
    ("Undiagnosed, HbA1c",               "undx",       "w_lab",  None),
    ("Screened in 3y, among undiagnosed", "screened",  "w_lab",  None),
    ("NCHS-style: Diagnosed",            "y_diag_fpg", "w_fast", None),
    ("NCHS-style: Total (HbA1c or FPG)", "y_bio_fpg",  "w_fast", None),
    ("NCHS-style: Undiagnosed",          "undx_fpg",   "w_fast", None),
]

rows = []
for cycle, d in df.groupby("cycle"):           # full cycle -> all PSUs stay in the design
    dof = d["psu"].nunique() - d["stratum"].nunique()      # FIX 2: design degrees of freedom
    t = stats.t.ppf(0.975, dof)
    for label, y, w, _ in MEASURES:
        p, se, n = svy_prop(d, y, w, d["adult"])
        rows.append({"cycle": cycle, "measure": label, "weight": w, "n": n, "dof": dof,
                     "pct": 100 * p, "se": 100 * se,
                     "ci_low": 100 * (p - t * se), "ci_high": 100 * (p + t * se)})
prev = pd.DataFrame(rows)
prev.to_csv(os.path.join(OUT_DIR, "step4_weighted_prevalence.csv"), index=False)
print("Weighted prevalence, adults 20+ (%):")
print(prev.pivot(index="measure", columns="cycle", values="pct").round(1).to_string())
print("\nDesign degrees of freedom:", prev.groupby("cycle")["dof"].first().to_dict())

Weighted prevalence, adults 20+ (%):
cycle                              2011-2012  2013-2014  2015-2016  2017-2020  2021-2023
measure                                                                                 
Diagnosed (y_diag)                       9.4       10.1       11.2       11.7       11.8
NCHS-style: Diagnosed                    9.2       10.0       11.1       11.7       11.3
NCHS-style: Total (HbA1c or FPG)        13.2       13.7       16.1       16.2       15.8
NCHS-style: Undiagnosed                  4.0        3.7        5.0        4.5        4.5
Screened in 3y, among undiagnosed       48.1       52.1       55.7       53.5       44.7
Total, HbA1c (y_bio)                    11.5       12.2       13.4       14.2       14.0
Undiagnosed, HbA1c                       2.2        2.0        2.0        2.4        2.3

Design degrees of freedom: {'2011-2012': 17, '2013-2014': 15, '2015-2016': 15, '2017-2020': 25, '2021-2023': 15}


## 4b. VALIDATION against the published NCHS numbers (FIX 3)

In [6]:
# NCHS Data Brief 516, Aug 2021–Aug 2023, adults 20+, crude %, pregnant excluded,
# fasting subsample weights. https://www.cdc.gov/nchs/products/databriefs/db516.htm
# To validate another cycle, add its crude values from the matching NCHS data brief.
NCHS = {
    ("2021-2023", "NCHS-style: Total (HbA1c or FPG)"): (15.8, 13.6, 18.2),
    ("2021-2023", "NCHS-style: Diagnosed"):            (11.3, 9.3, 13.5),
    ("2021-2023", "NCHS-style: Undiagnosed"):          (4.5, 3.3, 5.9),
}
val = []
for (cycle, measure), (pub, lo, hi) in NCHS.items():
    r = prev[(prev["cycle"] == cycle) & (prev["measure"] == measure)]
    if r.empty:
        continue
    ours = r["pct"].iloc[0]
    val.append({"cycle": cycle, "measure": measure, "ours_pct": round(ours, 1),
                "nchs_pct": pub, "nchs_ci": f"{lo}-{hi}",
                "match": "PASS" if lo <= ours <= hi else "FAIL"})
val = pd.DataFrame(val)
val.to_csv(os.path.join(OUT_DIR, "step4b_nchs_validation.csv"), index=False)
print(val.to_string(index=False))
if len(val) and (val["match"] == "PASS").all():
    print("\nVALIDATION PASSED — weights and design are wired correctly.")
else:
    print("\nVALIDATION NOT PASSED — fix this before anyone uses the data downstream.")

    cycle                          measure  ours_pct  nchs_pct   nchs_ci match
2021-2023 NCHS-style: Total (HbA1c or FPG)      15.8      15.8 13.6-18.2  PASS
2021-2023            NCHS-style: Diagnosed      11.3      11.3  9.3-13.5  PASS
2021-2023          NCHS-style: Undiagnosed       4.5       4.5   3.3-5.9  PASS

VALIDATION PASSED — weights and design are wired correctly.


## 5. Case counts per cycle x subgroup (power check)

In [7]:
MIN_EVENTS = 30
count_rows = []
adults = df[df["adult"] & df["undx"].notna()]
for cycle, d in adults.groupby("cycle"):
    for var in ["income_band", "insured", "routine_care", "age_band", "sex", "race_eth"]:
        for level, g in d.groupby(var, observed=True, dropna=False):
            count_rows.append({"cycle": cycle, "subgroup": var, "level": level,
                               "n_adults": len(g), "n_undiagnosed": int(g["undx"].sum()),
                               "n_screened_answered": int(g["screened"].notna().sum())})
counts = pd.DataFrame(count_rows)
counts["too_small"] = counts["n_undiagnosed"] < MIN_EVENTS
counts.to_csv(os.path.join(OUT_DIR, "step5_case_counts.csv"), index=False)
print(counts.pivot_table(index=["subgroup", "level"], columns="cycle",
                         values="n_undiagnosed", aggfunc="sum").to_string())
print(f"\n{int(counts['too_small'].sum())} cells have < {MIN_EVENTS} undiagnosed cases -> merge or pool.")

cycle                          2011-2012  2013-2014  2015-2016  2017-2020  2021-2023
subgroup     level                                                                  
age_band     20-39                    26         13         16         30         14
             40-59                    72         56         62        118         45
             60+                      71         79         73        136         93
income_band  1.3-3.5                  51         52         46         99         74
             <1.3                     66         51         60         66         29
             >3.5                     35         37         23         80         31
insured      0.0                      50         30         28         53         18
             1.0                     118        118        123        229        133
race_eth     Mexican American         21         24         35         44         19
             NH Asian                 30         24         20   